In [0]:
%pip install yfinance

In [0]:
import yfinance as yf
import time
from datetime import datetime

target_table_name = "finance_intelligence_hub.bronze.company_news_polygon"

# 1. جلب الشركات المتبقية (Resume Logic)
try:
    all_tickers_df = spark.sql("SELECT ticker FROM finance_intelligence_hub.bronze.companies")
    all_tickers = [row['ticker'] for row in all_tickers_df.collect()]
    
    try:
        # استبعاد الشركات اللي اتسحبت قبل كده عن طريق Yahoo Finance
        existing_tickers_df = spark.sql(f"SELECT DISTINCT ticker FROM {target_table_name} WHERE author = 'Yahoo Finance'")
        existing_tickers = set([row['ticker'] for row in existing_tickers_df.collect()])
    except Exception:
        existing_tickers = set()
        
    tickers_list = [t for t in all_tickers if t not in existing_tickers]
    print(f"📦 إجمالي الشركات: {len(all_tickers)} | المكتمل سابقاً: {len(existing_tickers)}")
    print(f"🚀 الشركات المتبقية وجاري سحبها من Yahoo Finance: {len(tickers_list)}")
except Exception as e:
    print(f"❌ خطأ كارثي في قراءة الجداول الأساسية: {e}")
    tickers_list = []

all_yahoo_records = []
total_saved_yahoo = 0

print(f"=== ⚡ بدء السحب النفاث والشامل من Yahoo Finance ===")

# 2. اللف على الشركات
for i, ticker in enumerate(tickers_list, 1):
    print(f"🔄 [{i}/{len(tickers_list)}] جاري جلب كامل أخبار: {ticker}...")
    
    try:
        # ربط الشركة بمكتبة yfinance
        ticker_obj = yf.Ticker(ticker)
        
        # جلب الأخبار
        news_list = ticker_obj.news
        
        if not news_list:
            time.sleep(0.5) # نوم خفيف حتى لو مفيش داتا لعدم الضغط
            continue
            
        for news in news_list:
            news_ts = news.get('providerPublishTime', 0)
            news_datetime = datetime.fromtimestamp(news_ts)
            
            if news_datetime.year < 2022:
                continue
                
            iso_date_str = news_datetime.strftime('%Y-%m-%dT%H:%M:%SZ')
            
            title = news.get('title', '')
            description = news.get('summary', '')
            if not description:
                description = title
                
            all_yahoo_records.append({
                "ticker": ticker,
                "published_date": iso_date_str,
                "title": title,
                "description": description,
                "text_for_finbert": f"{title}. {description}".strip(),
                "author": "Yahoo Finance",
                "source_url": news.get('link', ''),
                "image_url": news.get('thumbnail', {}).get('resolutions', [{}])[0].get('url', '') if news.get('thumbnail') else '',
                "publisher_name": news.get('publisher', 'Yahoo Finance'),
                "publisher_url": "",
                "related_tickers": ticker
            })
            
        # 3. حفظ الداتا كل ما نجمع 1000 خبر
        if len(all_yahoo_records) >= 1000:
            yahoo_df = spark.createDataFrame(all_yahoo_records)
            print(f"\n🔍 [معاينة داتا Yahoo] عينة من الدفعة الحالية:")
            yahoo_df.select("ticker", "published_date", "title").show(5, truncate=40)
            
            yahoo_df.write \
                .format("delta") \
                .mode("append") \
                .option("mergeSchema", "true") \
                .saveAsTable(target_table_name)
                
            total_saved_yahoo += len(all_yahoo_records)
            all_yahoo_records.clear()
            
        # نوم آمن بين الشركات لتجنب الـ Rate Limit
        time.sleep(0.5)
            
    except Exception as e:
        error_msg = str(e)
        # هنا المكان الصح للـ Rate Limit Handle جوه الـ Loop
        if "Too Many Requests" in error_msg or "Rate limited" in error_msg:
            print(f"\n⚠️ [Yahoo Rate Limit] ياهو قفل الطلبات! جاري النوم دقيقتين لتصفية الـ IP...")
            time.sleep(120)
            # إعادة إدراج الشركة عشان مانخسرهاش
            tickers_list.insert(i-1, ticker) 
            continue
        else:
            print(f"❌ خطأ أثناء سحب بيانات الشركة {ticker}: {e}")
            time.sleep(0.5)

# 4. حفظ الدفعة الأخيرة المتبقية
if all_yahoo_records:
    yahoo_df = spark.createDataFrame(all_yahoo_records)
    print(f"\n🔍 [معاينة داتا Yahoo] الدفعة الأخيرة:")
    yahoo_df.select("ticker", "published_date", "title").show(5, truncate=40)
    
    yahoo_df.write \
        .format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(target_table_name)
    total_saved_yahoo += len(all_yahoo_records)

print("\n" + "="*50)
print(f"🎉 تم إنهاء سحب Yahoo Finance بنجاح!")
print(f"📊 إجمالي الأخبار المضافة من ياهو: {total_saved_yahoo} خبر.")
print("="*50)

In [0]:
%sql
SELECT COUNT(*) FROM finance_intelligence_hub.bronze.company_news_polygon;